# 基於樹的模型 (Tree-Based Models)

這個 Notebook 將介紹幾種常用的基於樹的模型，包括：

1.  **決策樹 (Decision Tree)**
    -   1.1 分類樹
    -   1.2 回歸樹
2.  **隨機森林 (Random Forest)**
3.  **梯度提升樹 (Gradient Boosting Decision Trees, GBDT)**
    -   3.1 XGBoost
    -   3.2 LightGBM

我們將使用 `pandas`、`numpy`、`matplotlib`、`seaborn`、`scikit-learn`、`xgboost` 和 `lightgbm` 等套件來實作這些模型，並使用範例資料進行說明。

## 載入需要的套件

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score
from sklearn.datasets import load_iris, fetch_california_housing  # 更新：使用 California Housing 替代已棄用的 Boston
import xgboost as xgb
import lightgbm as lgb

# 設置隨機種子以確保結果可重現
np.random.seed(42)

print("套件載入完成！")
print("注意：本 Notebook 已更新為使用 California Housing dataset，替代已棄用的 Boston Housing dataset")

---

<a id='decision-tree'></a>
## 1. 決策樹 (Decision Tree)

決策樹是一種常見的機器學習演算法，可用於**分類**和**迴歸**問題。它通過遞迴地將資料劃分為不同的子集來建構樹狀模型，每個節點代表一個特徵，每個分支代表一個決策規則，每個葉節點代表一個輸出值 (類別標籤或連續值)。

**優點:**

-   易於理解和解釋 (可視覺化)。
-   可以處理數值型和類別型特徵。
-   不需要對資料進行特徵縮放。

**缺點:**

-   容易過度擬合。
-   對訓練資料中的小變化敏感。
-   可能找到的不是全域最佳解。

**建構決策樹的關鍵概念:**

-   **特徵選擇:**  在每個節點選擇最佳的特徵來劃分資料。常用的指標包括：
    -   **資訊增益 (Information Gain)** (ID3 演算法): 基於熵 (Entropy) 的概念，選擇資訊增益最大的特徵。
    -   **增益比率 (Gain Ratio)** (C4.5 演算法): 對資訊增益的改進，考慮了特徵的取值數量。
    -   **吉尼係數 (Gini Impurity)** (CART 演算法):  衡量資料的不純度，選擇吉尼係數最小的特徵。
-   **停止條件:**  決定何時停止樹的生長，例如：
    -   達到最大深度。
    -   節點中的樣本數少於某個閾值。
    -   節點的不純度低於某個閾值。
-   **剪枝 (Pruning):**  減少樹的複雜度，防止過度擬合。常見的剪枝方法包括：
    -   **預剪枝 (Pre-pruning):**  在樹的建構過程中，提前停止樹的生長。
    -   **後剪枝 (Post-pruning):**  在樹完全生長後，再剪去一些分支。

### 1.1 分類樹

In [ ]:
# 使用 scikit-learn 載入鳶尾花資料集
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

# 將資料劃分為訓練集和測試集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 建立決策樹分類器 (使用 Gini 係數)
clf = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42) 

# 訓練模型
clf.fit(X_train, y_train)

# 預測測試集
y_pred = clf.predict(X_test)

# 評估模型
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# 視覺化決策樹
plt.figure(figsize=(12, 8))
plot_tree(clf, filled=True, feature_names=iris.feature_names, class_names=iris.target_names)
plt.show()

[Image of Decision Tree]

### 1.2 回歸樹

In [ ]:
# 使用 California Housing dataset (替代已棄用的 Boston Housing)
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target

print(f"數據集大小: {X.shape}")
print(f"特徵名稱: {california.feature_names}")
print(f"目標變數: 房價中位數 (單位: $100,000)")

# 將資料劃分為訓練集和測試集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 建立決策樹回歸器 (使用 MSE)
reg = DecisionTreeRegressor(criterion='squared_error', max_depth=5, random_state=42)  # 增加深度以適應較複雜的數據

# 訓練模型
reg.fit(X_train, y_train)

# 預測測試集
y_pred = reg.predict(X_test)

# 評估模型
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"\nMean Squared Error (MSE): {mse:.4f}")
print(f"R-squared (R²): {r2:.4f}")

In [ ]:
# 視覺化決策樹 (限制顯示深度以提高可讀性)
plt.figure(figsize=(20, 10))
plot_tree(reg, filled=True, feature_names=california.feature_names, max_depth=3)  # 只顯示前3層
plt.title("Decision Tree Regressor (California Housing)", fontsize=16)
plt.tight_layout()
plt.show()

# 顯示特徵重要性
feature_importance = pd.DataFrame({
    'feature': california.feature_names,
    'importance': reg.feature_importances_
}).sort_values('importance', ascending=False)

print("\n特徵重要性排序:")
print(feature_importance)

[Image of Decision Tree Regressor]

---

<a id='random-forest'></a>
## 2. 隨機森林 (Random Forest)

隨機森林是一種**集成學習**方法，它通過建構多個決策樹並將它們的預測結果進行平均 (迴歸) 或投票 (分類) 來進行預測。隨機森林通常比單一決策樹具有更好的泛化能力，並且可以降低過度擬合的風險。

**隨機性體現在兩個方面:**

-   **樣本隨機:**  從訓練集中隨機抽取樣本 (Bootstrap 抽樣) 來訓練每個決策樹。
-   **特徵隨機:**  在每個節點分裂時，從所有特徵中隨機選擇一部分特徵來考慮。

**優點:**

-   通常具有較高的準確性。
-   可以處理大量的特徵。
-   可以估計特徵的重要性。
-   對異常值不太敏感。

**缺點:**

-   模型較為複雜，解釋性不如單一決策樹。
-   訓練時間可能較長，特別是當樹的數量較多時。

In [ ]:
# 建立隨機森林分類器
rf_clf = RandomForestClassifier(n_estimators=100, criterion='gini', max_depth=3, random_state=42)

# 使用鳶尾花資料集
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 訓練模型
rf_clf.fit(X_train, y_train)

# 預測測試集
y_pred = rf_clf.predict(X_test)

# 評估模型
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# 特徵重要性
feature_importances = pd.Series(rf_clf.feature_importances_, index=X.columns).sort_values(ascending=False)

# 繪製特徵重要性圖表
sns.barplot(x=feature_importances, y=feature_importances.index)
plt.xlabel('Feature Importance Score')
plt.ylabel('Features')
plt.title("Visualizing Important Features (Random Forest)")
plt.show()

[Image of Feature Importances (Random Forest)]

In [ ]:
# 建立隨機森林迴歸器
rf_reg = RandomForestRegressor(n_estimators=100, criterion='squared_error', max_depth=10, random_state=42, n_jobs=-1)  # n_jobs=-1 使用所有CPU核心

# 使用 California Housing dataset
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 訓練模型
rf_reg.fit(X_train, y_train)

# 預測測試集
y_pred = rf_reg.predict(X_test)

# 評估模型
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R²): {r2:.4f}")

# 可視化預測結果 vs 實際值
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("實際房價")
plt.ylabel("預測房價")
plt.title("Random Forest: 實際值 vs 預測值")
plt.tight_layout()
plt.show()

---

<a id='gradient-boosting'></a>
## 3. 梯度提升樹 (Gradient Boosting Decision Trees, GBDT)

梯度提升樹 (GBDT) 也是一種**集成學習**方法，它通過**循序地**建構多個決策樹來進行預測。每個新的決策樹都試圖修正前一個決策樹的預測誤差。GBDT 通常比隨機森林具有更好的準確性，但訓練時間也更長，且更容易過度擬合。

**核心思想:**

1.  建構一個初始的決策樹 (通常是一個簡單的樹)。
2.  計算當前模型的預測誤差 (殘差)。
3.  建構一個新的決策樹來預測殘差。
4.  將新的決策樹加到現有模型中，並更新預測結果。
5.  重複步驟 2-4，直到達到指定的樹的數量或誤差不再下降。

**優點:**

-   通常具有很高的準確性。
-   可以處理數值型和類別型特徵。
-   可以估計特徵的重要性。

**缺點:**

-   訓練時間可能較長，特別是當樹的數量較多時。
-   容易過度擬合，需要仔細調整參數。
-   模型較為複雜，解釋性不如單一決策樹。

### 3.1 XGBoost

XGBoost (Extreme Gradient Boosting) 是一個高效且流行的 GBDT 實作，它在原始 GBDT 演算法的基礎上進行了多項改進，包括：

-   **正則化:**  XGBoost 在目標函數中加入了正則化項，以控制模型的複雜度，防止過度擬合。
-   **二階導數:** XGBoost 使用了損失函數的二階導數 (Hessian 矩陣) 來加速訓練過程，並提高模型的準確性。
-   **缺失值處理:** XGBoost 可以自動處理缺失值。
-   **平行化:** XGBoost 支援平行化訓練，可以加快訓練速度。
-   **高效能:** XGBoost 經過了優化，具有很高的執行效率。

首先，您需要安裝 XGBoost 套件：

```bash
pip install xgboost
```

In [ ]:
# 建立 XGBoost 分類器
xgb_clf = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

# 使用鳶尾花資料集
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 訓練模型
xgb_clf.fit(X_train, y_train)

# 預測測試集
y_pred = xgb_clf.predict(X_test)

# 評估模型
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# 特徵重要性
feature_importances = pd.Series(xgb_clf.feature_importances_, index=X.columns).sort_values(ascending=False)

# 繪製特徵重要性圖表
sns.barplot(x=feature_importances, y=feature_importances.index)
plt.xlabel('Feature Importance Score')
plt.ylabel('Features')
plt.title("Visualizing Important Features (XGBoost)")
plt.show()

[Image of Feature Importances (XGBoost)]

In [ ]:
# 建立 XGBoost 迴歸器
xgb_reg = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, n_jobs=-1)

# 使用 California Housing dataset
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 訓練模型
xgb_reg.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

# 預測測試集
y_pred = xgb_reg.predict(X_test)

# 評估模型
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R²): {r2:.4f}")

### 3.2 LightGBM

LightGBM 是一個由微軟開發的 GBDT 框架，它在 XGBoost 的基礎上進行了進一步的優化，主要包括：

-   **基於直方圖的演算法:** LightGBM 使用基於直方圖的演算法來尋找最佳的節點分裂點，可以減少記憶體使用量和計算時間。
-   **帶有深度限制的葉子生長 (Leaf-wise) 策略:**  與 XGBoost 的按層生長 (Level-wise) 策略不同，LightGBM 採用帶有深度限制的葉子生長策略，可以更快地收斂，並降低過度擬合的風險。
-   **特徵並行和資料並行:** LightGBM 支援特徵並行和資料並行，可以進一步加快訓練速度。
-   **高效能:** LightGBM 通常比 XGBoost 訓練速度更快，記憶體使用量更低。

首先，您需要安裝 LightGBM 套件：

```bash
pip install lightgbm
```

In [ ]:
# 建立 LightGBM 分類器
lgb_clf = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

# 使用鳶尾花資料集
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 訓練模型
lgb_clf.fit(X_train, y_train)

# 預測測試集
y_pred = lgb_clf.predict(X_test)

# 評估模型
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# 特徵重要性
feature_importances = pd.Series(lgb_clf.feature_importances_, index=X.columns).sort_values(ascending=False)

# 繪製特徵重要性圖表
sns.barplot(x=feature_importances, y=feature_importances.index)
plt.xlabel('Feature Importance Score')
plt.ylabel('Features')
plt.title("Visualizing Important Features (LightGBM)")
plt.show()

[Image of Feature Importances (LightGBM)]

In [ ]:
# 建立 LightGBM 迴歸器
lgb_reg = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, n_jobs=-1, verbose=-1)

# 使用 California Housing dataset
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 訓練模型
lgb_reg.fit(X_train, y_train, eval_set=[(X_test, y_test)])

# 預測測試集
y_pred = lgb_reg.predict(X_test)

# 評估模型
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R²): {r2:.4f}")

# 比較三種模型的性能
print("\n" + "="*50)
print("模型性能比較摘要:")
print("="*50)
print("本節比較了決策樹、隨機森林、XGBoost 和 LightGBM 在回歸任務上的表現")
print("通常 XGBoost 和 LightGBM 會有最佳性能")
print("建議：在實際項目中使用交叉驗證來更準確地評估模型性能")

---

## 總結

這個 Notebook 介紹了幾種常用的基於樹的模型，包括決策樹、隨機森林和梯度提升樹 (XGBoost、LightGBM)。這些模型在各種機器學習任務中都表現出色，是您機器學習工具箱中不可或缺的一部分。

**選擇哪個模型?**

-   **決策樹:**  簡單易懂，可作為 baseline 模型，或用於需要模型解釋性的場景。
-   **隨機森林:**  通常比決策樹具有更好的準確性和泛化能力，適用於大多數情況。
-   **GBDT (XGBoost、LightGBM):**  通常具有最高的準確性，但需要更長的訓練時間和更多的參數調整。XGBoost 更為成熟穩定，LightGBM 則通常更快更輕量。

建議您根據具體的應用場景和資料特性來選擇合適的模型，並通過交叉驗證等方法來評估模型的效能，並調整模型的參數以獲得最佳結果。

In [ ]:
# 使用交叉驗證比較多個模型
from sklearn.model_selection import cross_val_score
import time

# 準備數據
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target

# 定義模型
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostRegressor(iterations=100, depth=5, learning_rate=0.1, random_state=42, verbose=0)
}

# 進行5折交叉驗證
results = {}
for name, model in models.items():
    start_time = time.time()
    scores = cross_val_score(model, X, y, cv=5, scoring='r2', n_jobs=-1)
    elapsed_time = time.time() - start_time
    
    results[name] = {
        'mean_r2': scores.mean(),
        'std_r2': scores.std(),
        'time': elapsed_time
    }
    
    print(f"{name}:")
    print(f"  R² Score: {scores.mean():.4f} (±{scores.std():.4f})")
    print(f"  訓練時間: {elapsed_time:.2f} 秒")
    print()

# 視覺化比較結果
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# R² Score 比較
model_names = list(results.keys())
r2_scores = [results[name]['mean_r2'] for name in model_names]
r2_stds = [results[name]['std_r2'] for name in model_names]

ax1.bar(model_names, r2_scores, yerr=r2_stds, capsize=5, alpha=0.7)
ax1.set_ylabel('R² Score')
ax1.set_title('模型性能比較 (5-Fold CV)')
ax1.set_ylim([0.7, 0.9])
ax1.grid(axis='y', alpha=0.3)

# 訓練時間比較
times = [results[name]['time'] for name in model_names]
ax2.bar(model_names, times, alpha=0.7, color='orange')
ax2.set_ylabel('訓練時間 (秒)')
ax2.set_title('訓練速度比較 (5-Fold CV)')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. 實戰技巧 (2024 Best Practices)

### 5.1 交叉驗證與模型評估

## 4. 模型比較與選擇指南 (2024 更新)

### 綜合比較表

| 模型 | 訓練速度 | 預測速度 | 準確度 | 記憶體使用 | 類別特徵支援 | 適用場景 |
|------|---------|---------|--------|-----------|-------------|---------|
| 決策樹 | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | 快速原型、可解釋性 |
| 隨機森林 | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | 穩定的基線模型 |
| XGBoost | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐ | Kaggle 競賽、通用場景 |
| LightGBM | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐ | 大規模數據、快速訓練 |
| **CatBoost** | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | **類別特徵多、生產部署** |

### 選擇建議

**1. 快速原型開發**
- 首選：**CatBoost** (開箱即用效果好)
- 次選：Random Forest

**2. Kaggle 競賽**
- 首選：**XGBoost + LightGBM + CatBoost 集成**
- 策略：嘗試所有三個並進行模型融合

**3. 生產環境部署**
- 首選：**CatBoost** (推理速度最快)
- 次選：LightGBM (記憶體佔用小)

**4. 大規模數據 (>1GB)**
- 首選：**LightGBM** (訓練速度快、記憶體效率高)
- 次選：CatBoost

**5. 包含大量類別特徵**
- 首選：**CatBoost** (原生支援類別特徵)
- 次選：手動編碼 + LightGBM/XGBoost

### 超參數調優建議

**CatBoost 推薦起始參數:**
```python
{
    'iterations': 1000,
    'learning_rate': 0.03,
    'depth': 6,
    'l2_leaf_reg': 3,
    'early_stopping_rounds': 50
}
```

**XGBoost 推薦起始參數:**
```python
{
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'early_stopping_rounds': 50
}
```

**LightGBM 推薦起始參數:**
```python
{
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'early_stopping_rounds': 50
}
```

In [ ]:
# CatBoost 回歸示例
from catboost import CatBoostRegressor

# 使用 California Housing dataset
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 建立 CatBoost 回歸器
catboost_reg = CatBoostRegressor(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    random_state=42,
    verbose=0
)

# 訓練模型
catboost_reg.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=10, verbose=False)

# 預測測試集
y_pred = catboost_reg.predict(X_test)

# 評估模型
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mse)

print("CatBoost 回歸結果:")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R²): {r2:.4f}")

In [ ]:
# CatBoost 分類示例
from catboost import CatBoostClassifier

# 使用鳶尾花資料集
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 建立 CatBoost 分類器
catboost_clf = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=5,
    random_state=42,
    verbose=0  # 關閉訓練日誌
)

# 訓練模型
catboost_clf.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=10, verbose=False)

# 預測測試集
y_pred = catboost_clf.predict(X_test)

# 評估模型
print("CatBoost 分類結果:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# 特徵重要性
feature_importances = pd.Series(catboost_clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\n特徵重要性:")
print(feature_importances)

In [ ]:
# 安裝並導入 CatBoost
try:
    from catboost import CatBoostRegressor, CatBoostClassifier, Pool
    print("CatBoost 已安裝！")
except ImportError:
    print("正在安裝 CatBoost...")
    import sys
    !{sys.executable} -m pip install catboost -q
    from catboost import CatBoostRegressor, CatBoostClassifier, Pool
    print("CatBoost 安裝完成！")

### 3.3 CatBoost (2024 推薦)

**CatBoost** 是由 Yandex 開發的梯度提升庫，特別擅長處理類別特徵。它在 XGBoost 和 LightGBM 的基礎上提供了幾個獨特優勢：

**主要特點:**

-   **原生支援類別特徵**: 無需手動進行 One-Hot 編碼或 Label Encoding
-   **對稱樹結構**: 減少過擬合風險
-   **Ordered Boosting**: 避免預測偏移 (Prediction Shift)
-   **開箱即用**: 預設參數通常就能達到很好的效果
-   **快速推理**: 推理速度比 XGBoost 和 LightGBM 更快
-   **內建交叉驗證和過擬合檢測**

**安裝:**
```bash
pip install catboost
```

**適用場景:**
- 表格數據競賽 (Kaggle, 天池等)
- 包含大量類別特徵的數據集
- 需要快速原型開發的場景
- 生產環境部署 (推理速度快)

## 參考資料

-   [Scikit-learn Documentation](https://scikit-learn.org/stable/)
-   [XGBoost Documentation](https://xgboost.readthedocs.io/)
-   [LightGBM Documentation](https://lightgbm.readthedocs.io/)